# Composing a Sentinel-2 NDVI pipeline

This notebook is the **operator-composition slice** of the canonical
Lake Tahoe scenario: cloud-free Sentinel-2 NDVI over Lake Tahoe, summer
2024. The end-to-end multi-repo flow (catalog discovery → patching →
operators) lives in
[`geocatalog/docs/notebooks/end_to_end_lake_tahoe.ipynb`](https://github.com/jejjohnson/geocatalog/blob/main/docs/notebooks/end_to_end_lake_tahoe.ipynb).
Here we focus on the geotoolz part: defining a few small `Operator`
subclasses and composing them with `Sequential` and `Graph`.

**Scenario.**

| | |
|---|---|
| AOI bbox (EPSG:4326) | `(-120.25, 38.85, -119.85, 39.30)` |
| Date range | `2024-06-01` to `2024-09-30` |
| STAC | Microsoft Planetary Computer (`planetarycomputer.microsoft.com/api/stac/v1`) |
| Collection | `sentinel-2-l2a` |
| Cloud cover | < 20 % |

## Pre-alpha note

Several `geotoolz` operator modules (`indices`, `cloud`, `radiometry`,
…) are landing module-by-module and their named-op surface is in flux.
To keep the patterns in this notebook stable, we define `Scale`,
`CloudMask`, `NDVI`, and `ApplyMask` **inline** as `Operator` subclasses.
This is also how a researcher iterates on a new operator before promoting
it into the package.

**This notebook is shipped un-executed** because the sandbox can't reach
Microsoft Planetary Computer. Run it locally with the dependencies
installed (`rioxarray`, `planetary-computer`, `pystac-client`,
`matplotlib`).

## 1. Load one Sentinel-2 scene via Microsoft Planetary Computer

We use `pystac-client` to find one low-cloud scene in the AOI / date
range, then `rioxarray` to load three bands: B04 (Red, 10 m), B08 (NIR,
10 m), and the SCL scene-classification band (20 m, resampled). The
result is wrapped in a `GeoTensor` for the pipeline.

In [ ]:
import numpy as np
import planetary_computer
import pystac_client
import rioxarray  # registers the .rio accessor
import xarray as xr
from georeader.geotensor import GeoTensor
from pipekit import Operator, Sequential

import geotoolz as gz

In [ ]:
BBOX = (-120.25, 38.85, -119.85, 39.30)  # Lake Tahoe
DATE_RANGE = "2024-06-01/2024-09-30"

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": 20}},
)
items = list(search.items())
print(f"found {len(items)} items")
item = items[0]
print(f"using {item.id}  cloud_cover={item.properties['eo:cloud_cover']:.1f}%")

In [ ]:
def _load(asset_key: str, *, resample_to: xr.DataArray | None = None) -> xr.DataArray:
    href = item.assets[asset_key].href
    da = rioxarray.open_rasterio(href, masked=True).squeeze("band", drop=True)
    da = da.rio.clip_box(*BBOX, crs="EPSG:4326")
    if resample_to is not None:
        da = da.rio.reproject_match(resample_to)
    return da


red = _load("B04")  # 10 m
nir = _load("B08", resample_to=red)  # 10 m, aligned to red
scl = _load("SCL", resample_to=red)  # 20 m → 10 m, aligned to red

stack = xr.concat([red, nir, scl], dim="band")  # (3, H, W)
stack["band"] = ["red", "nir", "scl"]

gt = GeoTensor(
    values=stack.values,
    transform=stack.rio.transform(),
    crs=stack.rio.crs,
)
print(f"GeoTensor shape={gt.values.shape}  crs={gt.crs}")

## 2. Define operators inline

Each operator follows the same two-method contract: `_apply` does the
work, `get_config` round-trips the constructor args. This is exactly the
shape you'd promote into a real `geotoolz.<module>` once the API
stabilises.

In [ ]:
class Scale(Operator):
    """DN → reflectance via a single scale factor.

    Sentinel-2 L2A is stored as DN with a 1e-4 scale to reflectance.
    This op is a toy stand-in for the real ``geotoolz.radiometry``
    operators (``DNToReflectance``, ``ApplyGain``, …).
    """

    def __init__(
        self, *, scale: float = 1e-4, band_idx: tuple[int, ...] = (0, 1)
    ) -> None:
        self.scale = scale
        self.band_idx = tuple(band_idx)

    def _apply(self, gt):
        out = gt.values.astype("float32").copy()
        for i in self.band_idx:
            out[i] = out[i] * self.scale
        return gt.array_as_geotensor(out)

    def get_config(self):
        return {"scale": self.scale, "band_idx": list(self.band_idx)}

In [ ]:
class CloudMask(Operator):
    """Boolean clear-pixel mask from the Sentinel-2 SCL band.

    Keeps SCL classes 4 (vegetation), 5 (not-vegetated), 6 (water),
    7 (unclassified), 11 (snow). Drops cloud / shadow / saturated.

    Output shape collapses the channel axis: (H, W).
    """

    KEEP = (4, 5, 6, 7, 11)

    def __init__(self, *, scl_idx: int = 2) -> None:
        self.scl_idx = scl_idx

    def _apply(self, gt):
        scl = gt.values[self.scl_idx]
        clear = np.isin(scl, self.KEEP).astype("uint8")
        return gt.array_as_geotensor(clear)

    def get_config(self):
        return {"scl_idx": self.scl_idx}

In [ ]:
class NDVI(Operator):
    """(NIR - Red) / (NIR + Red + eps); collapses channel axis."""

    def __init__(
        self, *, red_idx: int = 0, nir_idx: int = 1, eps: float = 1e-10
    ) -> None:
        self.red_idx, self.nir_idx, self.eps = red_idx, nir_idx, eps

    def _apply(self, gt):
        a = gt.values
        red, nir = a[self.red_idx], a[self.nir_idx]
        return gt.array_as_geotensor((nir - red) / (nir + red + self.eps))

    def get_config(self):
        return {"red_idx": self.red_idx, "nir_idx": self.nir_idx, "eps": self.eps}

In [ ]:
class ApplyMask(Operator):
    """Zero (or NaN) out pixels where mask == 0.

    Expects ``(reflectance_gt, mask_gt)`` as a tuple. ``mask_gt`` must
    have shape ``(H, W)`` matching the trailing dims of ``reflectance_gt``.
    """

    def __init__(self, *, fill: float = np.nan) -> None:
        self.fill = fill

    def _apply(self, inputs):
        gt, mask = inputs
        m = mask.values.astype(bool)
        out = gt.values.astype("float32").copy()
        out[:, ~m] = self.fill
        return gt.array_as_geotensor(out)

    def get_config(self):
        return {"fill": self.fill}

## 3. Compose with `Sequential`

The simplest possible shape — scale then NDVI, no cloud masking:

In [ ]:
pipe = Sequential(
    [
        Scale(scale=1e-4, band_idx=(0, 1)),
        NDVI(red_idx=0, nir_idx=1),
    ]
)
ndvi_gt = pipe(gt)
print("ndvi shape:", ndvi_gt.values.shape)
print("ndvi range:", float(np.nanmin(ndvi_gt.values)), float(np.nanmax(ndvi_gt.values)))

Same shape with the `|` pipe operator (inherited from `Operator`):

In [ ]:
pipe = Scale(scale=1e-4, band_idx=(0, 1)) | NDVI(red_idx=0, nir_idx=1)
print(pipe)
ndvi_gt = pipe(gt)

## 4. Visualise the output

Standard NDVI ramp (red-yellow-green), masked where reflectance is NaN.

In [ ]:
import matplotlib.pyplot as plt


fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(ndvi_gt.values, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
ax.set_title(f"Lake Tahoe NDVI — {item.id}")
ax.axis("off")
fig.colorbar(im, ax=ax, shrink=0.7, label="NDVI")
plt.show()

## 5. `Graph` variant — the same pipeline as a named DAG

When cloud masking enters the picture, the flow stops being linear. The
same scene needs to be split into (a) the reflectance bands (Scale) and
(b) the SCL mask (CloudMask); the two are fused by `ApplyMask`, then
NDVI runs on the clean reflectance.

Build it by calling operators on `Input` placeholders, then wrap with
`Graph(inputs=..., outputs=...)`:

In [ ]:
img = gz.Input("image")
scaled = Scale(scale=1e-4, band_idx=(0, 1))(img)
clear = CloudMask(scl_idx=2)(img)
clean = ApplyMask(fill=np.nan)((scaled, clear))
ndvi_node = NDVI(red_idx=0, nir_idx=1)(clean)

g = gz.Graph(
    inputs={"image": img},
    outputs={"ndvi": ndvi_node, "mask": clear, "scaled": scaled},
)
print(g)

result = g(image=gt)
ndvi_clean = result["ndvi"]
print(
    "clean ndvi range:",
    float(np.nanmin(ndvi_clean.values)),
    float(np.nanmax(ndvi_clean.values)),
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
ax1.imshow(result["mask"].values, cmap="gray")
ax1.set_title("Clear-pixel mask (white = clear)")
ax1.axis("off")
im2 = ax2.imshow(ndvi_clean.values, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
ax2.set_title("Cloud-masked NDVI")
ax2.axis("off")
fig.colorbar(im2, ax=ax2, shrink=0.7, label="NDVI")
plt.show()

## 6. When to use which composition primitive

The two cells above are deliberately the same pipeline expressed two
ways. The choice tree:

| You want… | Reach for | Why |
|---|---|---|
| A linear chain of 2–6 ops | `Sequential` (or `op_a \| op_b`) | Smallest API surface; flattens nested pipes. |
| Named intermediates / branching / fan-in | `Graph` | Same scene, multiple sub-pipelines, fused at the end. |
| Runtime two-way fork (e.g. reproject if geographic) | `Branch` | Predicate evaluates the whole carrier. |
| Sensor / product dispatch (`'S2'` vs `'L8'`) | `Switch` | Finite named cases; default is `Identity`. |
| One input → many named derived products | `Fanout` | Sugar over a single-input `Graph`. |

Each of these is itself an `Operator`, so they nest freely: a `Graph`
inside a `Sequential`, a `Switch` whose cases are `Sequential`s, a
`Fanout` whose values are `Graph`s.

In [ ]:
# A small Fanout — compute several derived products from the same scene.


class NDWI(Operator):
    """(Green - NIR) / (Green + NIR). For this scene we only have Red +
    NIR + SCL, so this is illustrative — swap green_idx in when you have
    a B03 band stacked too.
    """

    def __init__(
        self, *, green_idx: int = 0, nir_idx: int = 1, eps: float = 1e-10
    ) -> None:
        self.green_idx, self.nir_idx, self.eps = green_idx, nir_idx, eps

    def _apply(self, gt):
        a = gt.values
        g, n = a[self.green_idx], a[self.nir_idx]
        return gt.array_as_geotensor((g - n) / (g + n + self.eps))

    def get_config(self):
        return {"green_idx": self.green_idx, "nir_idx": self.nir_idx, "eps": self.eps}


scaled_gt = Scale(scale=1e-4, band_idx=(0, 1))(gt)
products = gz.Fanout(
    {
        "ndvi": NDVI(red_idx=0, nir_idx=1),
        "ndwi": NDWI(green_idx=0, nir_idx=1),  # illustrative only
    }
)(scaled_gt)
print("product keys:", list(products.keys()))

## 7. Where next

- **End-to-end Lake Tahoe (cross-repo):** the canonical multi-package
  walkthrough lives in
  [`geocatalog/docs/notebooks/end_to_end_lake_tahoe.ipynb`](https://github.com/jejjohnson/geocatalog/blob/main/docs/notebooks/end_to_end_lake_tahoe.ipynb)
  — STAC discovery → catalog loading → patching → operators.
- **More operator patterns:** the
  [pipeline idioms notebook](pipeline_idioms.ipynb) is a recipe gallery
  of observer / control-flow / QC patterns.
- **Deployment shapes:** the
  [deployment shapes notebook](deployment_shapes.ipynb) tours 13
  deployment patterns (notebook, ETL, FastAPI, tile server, …).
- **Concepts:** the [Concepts page](../concepts.md) explains the model
  behind these primitives with diagrams.
- **Recipes:** [Define an operator](../recipes/define-an-operator.md),
  [Branching pipelines](../recipes/branching-pipelines.md),
  [Integration with geocatalog & geopatcher](../recipes/integration-with-geocatalog-and-geopatcher.md).